In [1]:
import netket as nk
import netket.experimental as nkx
import numpy as np

# ======================== 步骤1：构建H₂的费米子希尔伯特空间（与案例一致） ========================
# 定义H₂的自旋轨道数（STO-3G基组，2个原子×2个自旋=4个自旋轨道）
n_orbitals = 4
n_electrons = (1, 1)  # 自旋向上、向下各1个电子（H₂自旋单重态）

# 构建费米子希尔伯特空间
hi = nkx.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    n_fermions=n_electrons,
    s=1/2  # 电子自旋
)

# ======================== 步骤2：定义费米子跳跃的连接图（与案例一致） ========================
# 连接图：仅允许0↔1、2↔3轨道对之间跳跃（自旋对称）
graph = nk.graph.Graph(edges=[(0,1), (2,3)])

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_8875/1414725550.py:11: DeprecationWarning: netket.experimental.hilbert.SpinOrbitalFermions is deprecated: use netket.hilbert.SpinOrbitalFermions (netket >= 3.12)
  hi = nkx.hilbert.SpinOrbitalFermions(
/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_8875/1414725550.py:11: DeprecationWarning: 
                          Declaring per-spin subsector constraint with `n_fermions` is deprecated.
                          Use `n_fermions_per_spin` instead.

                          As this class was experimental, this behaviour will break in NetKet 3.12 to be
                          released in early 2024.
                          
  hi = nkx.hilbert.SpinOrbitalF

In [3]:
# ======================== 步骤3：初始化费米子跳跃抽样器（核心） ========================
# MetropolisFermionHop的核心是内部的"提议规则"，这里初始化抽样器以获取提议逻辑
sampler = nk.sampler.MetropolisFermionHop(
    hilbert=hi,
    graph=graph,  # 约束跳跃的轨道对
    n_chains=1    # 单链（仅用于单个组态提议）
)

# ======================== 步骤4：从指定初始组态提议下一个组态 ========================
# 定义H₂的初始组态（如案例中的第一个组态：[0,1,0,1]）
initial_config = np.array([0, 1, 0, 1])
print(f"初始组态：{initial_config}")



初始组态：[0 1 0 1]


In [ ]:
sampler.init_state()

In [4]:
# 关键：将初始组态设置为抽样器的当前状态
# sampler.state是抽样器的内部状态，对应每条链的当前组态
sampler.state.init(n_chains=1)  # 初始化单链状态
sampler.state.chain = initial_config.reshape(1, -1)  # 设置单链的初始组态

# 核心操作：调用抽样器的"提议"方法，生成下一个组态
# proposal_generator是抽样器的内部提议生成器，返回（新组态, 接受概率因子）
proposal = sampler.proposal_generator(sampler.state)
proposed_config = proposal[0][0]  # 提取提议的新组态（单链，故取第0个）

# 输出结果
print(f"提议的新组态：{proposed_config}")



AttributeError: 'MetropolisSampler' object has no attribute 'state'

In [ ]:
# ======================== 步骤5：验证提议的合法性 ========================
# 验证1：总电子数不变（H₂必须为2个电子）
electron_count_old = np.sum(initial_config)
electron_count_new = np.sum(proposed_config)
print(f"\n合法性验证：")
print(f"初始组态电子数：{electron_count_old} | 提议组态电子数：{electron_count_new} → {'合法' if electron_count_old==electron_count_new else '非法'}")

# 验证2：跳跃仅发生在预定义的轨道对中
# 找出组态变化的轨道索引
changed_orbitals = np.where(initial_config != proposed_config)[0]
print(f"发生变化的轨道对：{changed_orbitals} → {'合法' if tuple(changed_orbitals) in graph.edges or tuple(reversed(changed_orbitals)) in graph.edges else '非法'}")